# 01 – Unity Catalog setup

Creates the medallion schemas and the volumes used in the project under the `workspace` catalog. All statements use `IF NOT EXISTS`, so the notebook can be re-run.

## Existing catalogs

Shows the catalogs available in Free Edition and the current catalog / schema.

Output: the first table lists the catalogs, the second shows the active catalog and schema.

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
SELECT current_catalog(), current_schema();

## Medallion schemas

Creates the bronze, silver and gold schemas under the `workspace` catalog.

`IF NOT EXISTS`: re-running the cell does not fail (idempotent).

Output: an empty result if there is no error; the check is in the next cell.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze COMMENT 'Raw Berka data as Delta tables';
CREATE SCHEMA IF NOT EXISTS workspace.silver COMMENT 'Current-state tables, CDC applied with MERGE';
CREATE SCHEMA IF NOT EXISTS workspace.gold   COMMENT 'Dimensional model, SCD Type 2 dim_customer';

## Schema check

Lists the schemas in the `workspace` catalog.

Output: `bronze`, `silver`, `gold` are listed, together with the built-in `default` and `information_schema`.

In [0]:
%sql
SHOW SCHEMAS IN workspace;

## Raw volume

Creates a managed volume named `raw` under `workspace.bronze`. The Berka files (8 tables) are uploaded here.

Managed volume: Databricks manages the physical location of the files; we only use the `/Volumes/workspace/bronze/raw/` path.

Output: "OK" if there is no error.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.bronze.raw COMMENT 'Landing zone for raw Berka files';

## Volume path check

Checks that the volume can be accessed by file path. The Berka files are read from this path later.

`dbutils.fs.ls()`: lists the files in a folder (Databricks utility).

Output: the volume is new, so the file count is 0. No error means the path is correct.

In [0]:
raw_volume_path = "/Volumes/workspace/bronze/raw/"

files = dbutils.fs.ls(raw_volume_path)

print("Path:", raw_volume_path)
print("File count:", len(files))

## Landing volume

Volume for the CDC batch files. `src/cdc_generator.py` (04_cdc_generate) writes the changes here as JSON lines: `landing/<table>/batch_<NNNN>/`.

Safe to re-run (`IF NOT EXISTS`). No error means the volume exists.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.bronze.landing
COMMENT 'CDC batch files (JSON lines), one folder per table and batch';

## Volume check

Lists the volumes under `workspace.bronze`. Output: two rows, `raw` and `landing`.

In [0]:
%sql
SHOW VOLUMES IN workspace.bronze;